# Stokastisk 0.1.0 — quickstart on Kaggle (2 × Tesla T4)

Settings: **Accelerator: GPU T4 × 2**, **Internet: ON**, Python 3.13. One line installs the SDK with the bundled vLLM 0.25.1, Triton attention and the CUDA 13.0 `sm_75` binaries (downloaded from the Hugging Face CDN, SHA256‑verified by pip). Nothing has to be uploaded as a dataset.

In [1]:
# 1. install (one line)
!pip install stokastisk --extra-index-url https://ai-biruni.github.io/stokastisk/simple/

Looking in indexes: https://pypi.org/simple, https://ai-biruni.github.io/stokastisk/simple/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.1/38.1 MB 54.9 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 43.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 66.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 61.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 48.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.9/44.9 MB 87.0 MB/s  0:00:006m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20/20 [stokastisk]0 [stokastisk]]ny]


In [2]:
# 2. check the installation (no GPU engine is started here)
import stokastisk, json
print('stokastisk', stokastisk.__version__)
print(json.dumps(stokastisk.prohibited_report(), indent=1))   # clean = no FlashAttention / FlashInfer / FlashMLA anywhere
print(stokastisk.check_model('Qwen/Qwen2.5-0.5B-Instruct'))

stokastisk 0.1.0
{
 "distributions": [],
 "modules": [],
 "files": [],
 "native_hits": {},
 "binaries": [],
 "clean": true
}
{'model': 'Qwen/Qwen2.5-0.5B-Instruct', 'dtype': 'half', 'quantization': {'method': None, 'status': 'SUPPORTED', 'evidence': 'kaggle', 'note': 'no quantization (all Kaggle runs)'}, 'architectures': [], 'note': 'architecture unknown: pass architectures=[...] or a local model directory (no network access is used)', 'status': 'UNKNOWN_REQUIRES_EVIDENCE'}


In [3]:
%%writefile demo.py
# vLLM starts its engine in a separate process (spawn): keep the engine code in a script with a __main__ guard.
import stokastisk

if __name__ == '__main__':
    schema = {'type': 'object', 'properties': {'city': {'type': 'string', 'maxLength': 40}, 'population': {'type': 'integer', 'minimum': 0, 'maximum': 100000000}},
              'required': ['city', 'population'], 'additionalProperties': False}   # bound open-ended fields: a grammar cannot make a model stop
    with stokastisk.LLM('Qwen/Qwen2.5-0.5B-Instruct', config=stokastisk.StokastiskConfig(max_model_len=2048)) as llm:
        print('attention backend :', llm.verify_attention_backend())           # TRITON_ATTN
        print('completion        :', llm.complete('The capital of France is', max_tokens=16))
        print('chat              :', llm.chat([{'role': 'user', 'content': 'Say hello in five words.'}])[0].outputs[0].text)
        print('structured        :', llm.generate_structured(['Give a JSON object describing Paris.'], schema, max_tokens=96))

Writing demo.py


In [4]:
!python demo.py

INFO 11-13 16:49:14 [api_utils.py:273] non-default args: {'dtype': 'half', 'max_model_len': 2048, 'gpu_memory_utilization': 0.88, 'disable_log_stats': True, 'enforce_eager': True, 'disable_custom_all_reduce': True, 'structured_outputs_config': StructuredOutputsConfig(backend='xgrammar', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), 'attention_config': AttentionConfig(backend=<AttentionBackendEnum.TRITON_ATTN: 'vllm.v1.attention.backends.triton_attn.TritonAttentionBackend'>, flash_attn_version=None, use_prefill_decode_attention=False, flash_attn_max_num_splits_for_cuda_graph=32, tq_max_kv_splits_for_cuda_graph=32, use_trtllm_attention=None, disable_flashinfer_q_quantization=False, mla_prefill_backend=None, use_prefill_query_quantization=False, use_fp4_indexer_cache=False, indexer_kv_dtype='bf16', use_non_causal=False, flex_attn_block_m=None, flex_attn_block_n=None, flex_attn_q_block_size=No

## More
* Two GPUs: `StokastiskConfig(tensor_parallel_size=2)` (a model too big for one T4) or `stokastisk.parallel.ReplicaPool(model, devices=[0, 1])` (two independent copies). For a model that fits one T4, TP=1 is fastest.
* OpenAI‑compatible server: `with stokastisk.serving.ServerProcess('Qwen/Qwen2.5-0.5B-Instruct') as srv: srv.wait_ready(); ...` — see the README.
* Unsupported: bfloat16, FlashAttention/FlashInfer/FlashMLA, FP8/FP4, Mistral‑family models.